<a href="https://colab.research.google.com/github/betulcelik1/NLP_Project/blob/main/notebooks/02_Group_CrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import subprocess

repo_dir = "/content/NLP_Project"

if not os.path.isdir(os.path.join(repo_dir, ".git")):
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/betulcelik1/NLP_Project.git",
            repo_dir
        ],
        check=True
    )

os.chdir(repo_dir)

print("Project directory:", os.getcwd())
print("Files:", os.listdir(repo_dir))

Project directory: /content/NLP_Project
Files: ['notebooks', '.git', 'README.md', 'experiments', 'DiCoNLI_First.ipynb']


In [2]:
import pandas as pd
from pathlib import Path

csv_files = list(Path("/content/NLP_Project").rglob("*.csv"))

print("CSV files found:")
for file in csv_files:
    print(file)

train_candidates = [
    file for file in csv_files
    if "train" in file.name.lower()
    and "track1" in file.name.lower()
]

print("\nTraining candidates:")
for file in train_candidates:
    print(file)

if len(train_candidates) == 1:
    train_df = pd.read_csv(train_candidates[0])
    print("\nTraining data loaded:", train_df.shape)
    display(train_df.head())
else:
    print("\nTraining file needs to be identified.")

CSV files found:

Training candidates:

Training file needs to be identified.


In [5]:
import os
import subprocess

official_repo = "/content/SemEval-2027-Task-2-DiCo-NLI"

repo_url = (
    "https://github.com/ilopezgazpio/"
    "SemEval-2027-Task-2-DiCo-NLI.git"
)

result = subprocess.run(
    ["git", "clone", repo_url, official_repo],
    capture_output=True,
    text=True
) if not os.path.isdir(os.path.join(official_repo, ".git")) else None

if result is not None:
    print("Return code:", result.returncode)
    print(result.stderr[-1000:])

print("Repository ready:",
      os.path.isdir(os.path.join(official_repo, ".git")))

if os.path.isdir(official_repo):
    print("Files:", os.listdir(official_repo)[:15])

Return code: 0
Cloning into '/content/SemEval-2027-Task-2-DiCo-NLI'...

Repository ready: True
Files: ['References', '_config.yml', 'assets', 'latex', 'starter_kit', '.gitignore', 'advertising', 'final_data', 'trial_data', 'static', '.git', 'README.md', 'LICENSE', 'evaluation_functions', '.github']


In [6]:
import pandas as pd
from pathlib import Path

train_path = (
    Path(official_repo)
    / "final_data/train"
    / "dico_nli_train_track1_participant_labeled.csv"
)

print("Training file exists:", train_path.exists())

if train_path.exists():
    train_df = pd.read_csv(train_path)

    print("Train shape:", train_df.shape)
    print("Columns:", train_df.columns.tolist())
    display(train_df.head(3))

Training file exists: True
Train shape: (3042, 7)
Columns: ['instance_id', 'pair_id', 'text1_lang', 'text2_lang', 'text1', 'text2', 'label']


,instance_id,pair_id,text1_lang,text2_lang,text1,text2,label
0,dico_phrasis_0000001__en-en__flipped,dico_phrasis_0000001,en,en,Conservative opposition,Conservatives,FORWARD_ENTAILMENT
1,dico_phrasis_0000001__en-en__original,dico_phrasis_0000001,en,en,Conservatives,Conservative opposition,BACKWARD_ENTAILMENT
2,dico_phrasis_0000003__en-en__flipped,dico_phrasis_0000003,en,en,Japanese PM Shinzo Abe,Abe,FORWARD_ENTAILMENT


In [7]:
from sklearn.model_selection import GroupKFold
import numpy as np

def normalize_text(s):
    return " ".join(str(s).lower().split())

def make_pair_key(df):
    return df.apply(
        lambda row: tuple(sorted([
            normalize_text(row["text1"]),
            normalize_text(row["text2"])
        ])),
        axis=1
    )

parent = list(range(len(train_df)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_pair_ids = {}
seen_text_pairs = {}

text_keys = make_pair_key(train_df).tolist()

for i, (pair_id, text_key) in enumerate(
    zip(train_df["pair_id"], text_keys)
):
    if pair_id in seen_pair_ids:
        union(i, seen_pair_ids[pair_id])
    else:
        seen_pair_ids[pair_id] = i

    if text_key in seen_text_pairs:
        union(i, seen_text_pairs[text_key])
    else:
        seen_text_pairs[text_key] = i

groups = np.array([
    find(i) for i in range(len(train_df))
])

cv = GroupKFold(n_splits=5)

folds = list(
    cv.split(
        train_df,
        train_df["label"],
        groups=groups
    )
)

print("Total samples:", len(train_df))
print("Unique groups:", len(set(groups)))
print("Number of folds:", len(folds))

Total samples: 3042
Unique groups: 1725
Number of folds: 5


In [8]:
for fold_no, (train_idx, val_idx) in enumerate(folds, 1):

    fold_train = train_df.iloc[train_idx]
    fold_val = train_df.iloc[val_idx]

    pair_overlap = (
        set(fold_train["pair_id"]) &
        set(fold_val["pair_id"])
    )

    text_overlap = (
        set(make_pair_key(fold_train)) &
        set(make_pair_key(fold_val))
    )

    print(f"\nFold {fold_no}")
    print("Train:", len(fold_train))
    print("Validation:", len(fold_val))
    print("Overlapping pair IDs:", len(pair_overlap))
    print("Overlapping text pairs:", len(text_overlap))

    print("Validation labels:")
    print(fold_val["label"].value_counts().to_dict())


Fold 1
Train: 2433
Validation: 609
Overlapping pair IDs: 0
Overlapping text pairs: 0
Validation labels:
{'FORWARD_ENTAILMENT': 175, 'BACKWARD_ENTAILMENT': 175, 'EQUIVALENCE': 162, 'NEGATIVE_OTHER': 97}

Fold 2
Train: 2433
Validation: 609
Overlapping pair IDs: 0
Overlapping text pairs: 0
Validation labels:
{'FORWARD_ENTAILMENT': 176, 'BACKWARD_ENTAILMENT': 176, 'EQUIVALENCE': 162, 'NEGATIVE_OTHER': 95}

Fold 3
Train: 2434
Validation: 608
Overlapping pair IDs: 0
Overlapping text pairs: 0
Validation labels:
{'FORWARD_ENTAILMENT': 179, 'BACKWARD_ENTAILMENT': 179, 'EQUIVALENCE': 152, 'NEGATIVE_OTHER': 98}

Fold 4
Train: 2434
Validation: 608
Overlapping pair IDs: 0
Overlapping text pairs: 0
Validation labels:
{'FORWARD_ENTAILMENT': 175, 'BACKWARD_ENTAILMENT': 175, 'EQUIVALENCE': 164, 'NEGATIVE_OTHER': 94}

Fold 5
Train: 2434
Validation: 608
Overlapping pair IDs: 0
Overlapping text pairs: 0
Validation labels:
{'FORWARD_ENTAILMENT': 176, 'BACKWARD_ENTAILMENT': 176, 'EQUIVALENCE': 162, 'NEGATI

In [9]:
import re
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

def lexical_features(df):
    features = []

    for t1, t2 in zip(df["text1"], df["text2"]):
        a = set(re.findall(r"\b\w+\b", str(t1).lower()))
        b = set(re.findall(r"\b\w+\b", str(t2).lower()))

        intersection = len(a & b)

        overlap1 = intersection / len(a) if a else 0
        overlap2 = intersection / len(b) if b else 0
        length_diff = (
            abs(len(a) - len(b)) / max(len(a), len(b))
            if max(len(a), len(b)) else 0
        )

        features.append([overlap1, overlap2, length_diff])

    return np.array(features, dtype=float)


def build_exp05_features(train_part, val_part):
    train_matrices = []
    val_matrices = []

    for column, analyzer, ngrams, max_features, min_df in [
        ("text1", "word", (1, 2), 50000, 1),
        ("text2", "word", (1, 2), 50000, 1),
        ("text1", "char", (2, 5), 10000, 2),
        ("text2", "char", (2, 5), 10000, 2)
    ]:
        vectorizer = TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngrams,
            max_features=max_features,
            min_df=min_df
        )

        train_matrices.append(
            vectorizer.fit_transform(
                train_part[column].fillna("")
            )
        )

        val_matrices.append(
            vectorizer.transform(
                val_part[column].fillna("")
            )
        )

    X_train = hstack(
        train_matrices + [
            csr_matrix(lexical_features(train_part))
        ]
    ).tocsr()

    X_val = hstack(
        val_matrices + [
            csr_matrix(lexical_features(val_part))
        ]
    ).tocsr()

    return X_train, X_val

print("EXP-05 feature builder ready.")

EXP-05 feature builder ready.


In [10]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

cv_results = []
oof_predictions = pd.Series(index=train_df.index, dtype="object")

for fold_no, (train_idx, val_idx) in enumerate(folds, 1):

    fold_train = train_df.iloc[train_idx]
    fold_val = train_df.iloc[val_idx]

    X_tr, X_val = build_exp05_features(
        fold_train, fold_val
    )

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_tr, fold_train["label"])

    predictions = model.predict(X_val)
    oof_predictions.iloc[val_idx] = predictions

    scores = {
        "Fold": fold_no,
        "Accuracy": accuracy_score(
            fold_val["label"], predictions
        ),
        "Macro F1": f1_score(
            fold_val["label"], predictions,
            average="macro"
        ),
        "Weighted F1": f1_score(
            fold_val["label"], predictions,
            average="weighted"
        )
    }

    cv_results.append(scores)

    print(
        f"Fold {fold_no}/5 completed | "
        f"Weighted F1: {scores['Weighted F1']:.4f}"
    )

results_df = pd.DataFrame(cv_results)

print("\n=== 5-FOLD RESULTS ===")
display(results_df.round(4))

print("\n=== MEAN ===")
print(results_df.mean(numeric_only=True).round(4))

print("\n=== STANDARD DEVIATION ===")
print(
    results_df[
        ["Accuracy", "Macro F1", "Weighted F1"]
    ].std().round(4)
)

print("\nOOF predictions missing:",
      oof_predictions.isna().sum())

Fold 1/5 completed | Weighted F1: 0.6386
Fold 2/5 completed | Weighted F1: 0.5928
Fold 3/5 completed | Weighted F1: 0.6198
Fold 4/5 completed | Weighted F1: 0.6343
Fold 5/5 completed | Weighted F1: 0.5634

=== 5-FOLD RESULTS ===


,Fold,Accuracy,Macro F1,Weighted F1
0,1,0.6355,0.6151,0.6386
1,2,0.5944,0.5639,0.5928
2,3,0.6118,0.5923,0.6198
3,4,0.6316,0.6029,0.6343
4,5,0.5526,0.5307,0.5634



=== MEAN ===
Fold           3.0000
Accuracy       0.6052
Macro F1       0.5810
Weighted F1    0.6098
dtype: float64

=== STANDARD DEVIATION ===
Accuracy       0.0337
Macro F1       0.0339
Weighted F1    0.0315
dtype: float64

OOF predictions missing: 0


In [12]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)
import pandas as pd

labels = [
    "BACKWARD_ENTAILMENT",
    "EQUIVALENCE",
    "FORWARD_ENTAILMENT",
    "NEGATIVE_OTHER"
]

print("=== OOF Classification Report ===")

print(classification_report(
    train_df["label"],
    oof_predictions,
    labels=labels,
    digits=4
))

cm = confusion_matrix(
    train_df["label"],
    oof_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("\n=== Confusion Matrix ===")
display(cm_df)

=== OOF Classification Report ===
                     precision    recall  f1-score   support

BACKWARD_ENTAILMENT     0.7245    0.7015    0.7128       881
        EQUIVALENCE     0.5443    0.5212    0.5325       802
 FORWARD_ENTAILMENT     0.7337    0.6879    0.7100       881
     NEGATIVE_OTHER     0.3345    0.4163    0.3709       478

           accuracy                         0.6052      3042
          macro avg     0.5842    0.5817    0.5816      3042
       weighted avg     0.6183    0.6052    0.6107      3042


=== Confusion Matrix ===


,BACKWARD_ENTAILMENT,EQUIVALENCE,FORWARD_ENTAILMENT,NEGATIVE_OTHER
BACKWARD_ENTAILMENT,618,104,43,116
EQUIVALENCE,114,418,111,159
FORWARD_ENTAILMENT,44,110,606,121
NEGATIVE_OTHER,77,136,66,199


In [13]:
from sklearn.metrics import classification_report

train_idx_5, val_idx_5 = folds[4]

fold5_df = train_df.iloc[val_idx_5].copy()
fold5_df["predicted_label"] = (
    oof_predictions.iloc[val_idx_5].values
)

fold5_errors = fold5_df[
    fold5_df["label"] != fold5_df["predicted_label"]
]

print("Fold 5 total samples:", len(fold5_df))
print("Fold 5 errors:", len(fold5_errors))

print("\n=== Fold 5 Classification Report ===")
print(classification_report(
    fold5_df["label"],
    fold5_df["predicted_label"],
    digits=4
))

print("\n=== Fold 5 Most Common Errors ===")

error_types = (
    fold5_errors
    .groupby(["label", "predicted_label"])
    .size()
    .sort_values(ascending=False)
)

display(error_types.head(10))

print("\n=== Example Errors ===")

display(
    fold5_errors[
        ["text1", "text2", "label", "predicted_label"]
    ].head(10)
)

Fold 5 total samples: 608
Fold 5 errors: 272

=== Fold 5 Classification Report ===
                     precision    recall  f1-score   support

BACKWARD_ENTAILMENT     0.6962    0.6250    0.6587       176
        EQUIVALENCE     0.5063    0.4938    0.5000       162
 FORWARD_ENTAILMENT     0.7019    0.6420    0.6706       176
     NEGATIVE_OTHER     0.2519    0.3511    0.2933        94

           accuracy                         0.5526       608
          macro avg     0.5391    0.5280    0.5307       608
       weighted avg     0.5786    0.5526    0.5634       608


=== Fold 5 Most Common Errors ===


,,0
label,predicted_label,
EQUIVALENCE,NEGATIVE_OTHER,34
BACKWARD_ENTAILMENT,NEGATIVE_OTHER,33
FORWARD_ENTAILMENT,NEGATIVE_OTHER,31
NEGATIVE_OTHER,EQUIVALENCE,28
FORWARD_ENTAILMENT,EQUIVALENCE,25
EQUIVALENCE,FORWARD_ENTAILMENT,25
BACKWARD_ENTAILMENT,EQUIVALENCE,25
EQUIVALENCE,BACKWARD_ENTAILMENT,23
NEGATIVE_OTHER,BACKWARD_ENTAILMENT,18



=== Example Errors ===


,text1,text2,label,predicted_label
0,Conservative opposition,Conservatives,FORWARD_ENTAILMENT,NEGATIVE_OTHER
1,Conservatives,Conservative opposition,BACKWARD_ENTAILMENT,NEGATIVE_OTHER
7,among 9 killed,among all dead,BACKWARD_ENTAILMENT,NEGATIVE_OTHER
10,Landslide Win,'s win,FORWARD_ENTAILMENT,NEGATIVE_OTHER
11,'s win,Landslide Win,BACKWARD_ENTAILMENT,NEGATIVE_OTHER
100,16 soldiers,12 Soldiers,FORWARD_ENTAILMENT,NEGATIVE_OTHER
101,12 Soldiers,16 soldiers,BACKWARD_ENTAILMENT,NEGATIVE_OTHER
110,Canadian government,Canada,FORWARD_ENTAILMENT,BACKWARD_ENTAILMENT
111,Canada,Canadian government,BACKWARD_ENTAILMENT,FORWARD_ENTAILMENT
174,bubbles,a bubble,FORWARD_ENTAILMENT,EQUIVALENCE
